In [4]:

"""
fraud_detection.py

Fraud Detection Using Transaction Patterns
 
Tools: Python, Pandas, SQL (SQLite), Scikit-learn, Matplotlib
 
Steps
  1. Load transactions with Pandas
  2. Prepare features using SQL (how much, how often, when)
  3. Train an Isolation Forest to find unusual transactions
  4. Show risky transactions using Matplotlib charts
"""
 
import argparse
import os
import sqlite3
 
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
 
DATA_PATH = "bank_transactions_50K.csv"  # default; override with --data
 
# If your file uses different column names, map them here.
# Format: "your column name": "required name"
# Required: customer_id, txn_time, amount
# Optional: txn_id (created if missing), is_fraud (only used for evaluation)

COLUMN_MAP = {
       "TransactionID": "txn_id",
       "CustomerID": "customer_id",
       "TransactionDate": "txn_time",
       "TransactionAmount": "amount",
       "IsFraud": "is_fraud"
   }
 
OUTPUT_DIR = "outputs"
CONTAMINATION = 0.01  # expected share of risky transactions (about 2%)
 
 
# ----------------------------------------------------------------------
# 1. LOAD DATA (Pandas) + STORE IN A SQL TABLE
# ----------------------------------------------------------------------
def load_data_to_sql(path):
    df = pd.read_csv(path).rename(columns=COLUMN_MAP)
 
    # Check that the required columns are present
    required = ["customer_id", "txn_time", "amount"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing column(s): {missing}. Your file has: {list(df.columns)}. "
            "Rename them or edit COLUMN_MAP at the top of this file."
        )
 
    # Basic cleaning
    df["txn_time"] = pd.to_datetime(df["txn_time"], errors="coerce")
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    before = len(df)
    df = df.dropna(subset=required)
    df = df[df["amount"] > 0]
    if len(df) < before:
        print(f"Removed {before - len(df)} rows with missing/invalid values")
 
    # Optional columns
    if "txn_id" not in df.columns:
        df["txn_id"] = np.arange(1, len(df) + 1)
    if "is_fraud" not in df.columns:
        df["is_fraud"] = np.nan  # no labels available
 
    df = df[["txn_id", "customer_id", "txn_time", "amount", "is_fraud"]].copy()
    df["txn_time"] = df["txn_time"].dt.strftime("%Y-%m-%d %H:%M:%S")
 
    conn = sqlite3.connect(":memory:")  # temporary in-memory database
    df.to_sql("transactions", conn, index=False, if_exists="replace")
    return conn
 
 
# ----------------------------------------------------------------------
# 2. FEATURE PREPARATION (SQL)
#    How much -> amount, amount_ratio
#    How often -> daily_txn_count
#    When     -> hour
# ----------------------------------------------------------------------
FEATURE_QUERY = """
SELECT
    t.txn_id,
    t.customer_id,
    t.txn_time,
    t.amount,                                              -- how much
    ROUND(t.amount / c.avg_amount, 2)  AS amount_ratio,    -- vs customer's normal
    d.daily_txn_count,                                     -- how often
    CAST(strftime('%H', t.txn_time) AS INTEGER) AS hour,   -- when
    t.is_fraud
FROM transactions t
JOIN (
    SELECT customer_id, AVG(amount) AS avg_amount
    FROM transactions
    GROUP BY customer_id
) c ON t.customer_id = c.customer_id
JOIN (
    SELECT customer_id, DATE(txn_time) AS txn_date, COUNT(*) AS daily_txn_count
    FROM transactions
    GROUP BY customer_id, DATE(txn_time)
) d ON t.customer_id = d.customer_id
   AND DATE(t.txn_time) = d.txn_date
ORDER BY t.txn_time;
"""
 
 
def prepare_features(conn):
    df = pd.read_sql_query(FEATURE_QUERY, conn, parse_dates=["txn_time"])
    return df
 
 
# ----------------------------------------------------------------------
# 3. MODEL (Scikit-learn Isolation Forest)
# ----------------------------------------------------------------------
FEATURES = ["amount", "amount_ratio", "daily_txn_count", "hour"]
 
 
def detect_anomalies(df):
    X = StandardScaler().fit_transform(df[FEATURES])
 
    model = IsolationForest(
        n_estimators=200,
        contamination=CONTAMINATION,
        random_state=42,
    )
    model.fit(X)
 
    # predict() gives -1 for anomalies and 1 for normal transactions
    df["is_anomaly"] = (model.predict(X) == -1).astype(int)
    # lower score = more unusual, so flip the sign to make a "risk score"
    df["risk_score"] = -model.score_samples(X)
    return df
 
 
# ----------------------------------------------------------------------
# 4. CHARTS (Matplotlib)
# ----------------------------------------------------------------------
def make_charts(df):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    normal = df[df["is_anomaly"] == 0]
    risky = df[df["is_anomaly"] == 1]
 
    # Chart 1: Amount vs time of day
    plt.figure(figsize=(9, 5))
    plt.scatter(normal["hour"], normal["amount"], s=10, alpha=0.4,
                color="steelblue", label="Normal")
    plt.scatter(risky["hour"], risky["amount"], s=25, color="crimson",
                label="Risky")
    plt.yscale("log")
    plt.xlabel("Hour of day")
    plt.ylabel("Transaction amount (log scale)")
    plt.title("Transaction Amount vs Hour of Day")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/1_amount_vs_hour.png", dpi=150)
    plt.close()
 
    # Chart 2: Risky transactions by hour
    plt.figure(figsize=(9, 5))
    risky["hour"].value_counts().reindex(range(24), fill_value=0).plot(
        kind="bar", color="crimson")
    plt.xlabel("Hour of day")
    plt.ylabel("Number of risky transactions")
    plt.title("When Do Risky Transactions Happen?")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/2_risky_by_hour.png", dpi=150)
    plt.close()
 
    # Chart 3: Amount distribution, normal vs risky
    plt.figure(figsize=(9, 5))
    bins = range(0, int(df["amount"].quantile(0.995)) + 1000, 1000)
    plt.hist(normal["amount"], bins=bins, alpha=0.6, color="steelblue",
             label="Normal")
    plt.hist(risky["amount"], bins=bins, alpha=0.8, color="crimson",
             label="Risky")
    plt.xlabel("Transaction amount")
    plt.ylabel("Number of transactions")
    plt.title("Amount Distribution: Normal vs Risky")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/3_amount_distribution.png", dpi=150)
    plt.close()
 
    # Chart 4: Top 10 riskiest transactions
    top = df.nlargest(10, "risk_score").sort_values("risk_score")
    plt.figure(figsize=(9, 5))
    plt.barh(top["txn_id"].astype(str), top["risk_score"], color="crimson")
    plt.xlabel("Risk score (higher = more unusual)")
    plt.ylabel("Transaction ID")
    plt.title("Top 10 Riskiest Transactions")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/4_top_risky_transactions.png", dpi=150)
    plt.close()
 
 
# ----------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser(description="Transaction fraud detection")
    parser.add_argument("--data", default=DATA_PATH, help="path to your CSV file")
    args, _ = parser.parse_known_args()  # ignore extra args (e.g. from Jupyter)
 
    conn = load_data_to_sql(args.data)
    df = prepare_features(conn)
    df = detect_anomalies(df)
 
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    flagged = df[df["is_anomaly"] == 1].sort_values("risk_score", ascending=False)
    flagged.to_csv(f"{OUTPUT_DIR}/flagged_transactions.csv", index=False)
 
    print(f"Total transactions : {len(df)}")
    print(f"Flagged as risky   : {len(flagged)} ({len(flagged) / len(df):.1%})")
    print("\nTop 5 riskiest transactions:")
    print(flagged[["txn_id", "customer_id", "txn_time", "amount",
                   "hour", "risk_score"]].head().to_string(index=False))
 
    # If the file has true labels (e.g. the simulated data), check the model.
    if df["is_fraud"].notna().all():
        print("\nModel check against known labels:")
        print(classification_report(df["is_fraud"], df["is_anomaly"],
                                    target_names=["Normal", "Fraud"], digits=2))
 
    make_charts(df)
    print(f"Charts and flagged list saved in '{OUTPUT_DIR}/'")
    conn.close()
 
 
if __name__ == "__main__":
    main()
 



Removed 85 rows with missing/invalid values
Total transactions : 49919
Flagged as risky   : 500 (1.0%)

Top 5 riskiest transactions:
 txn_id  customer_id            txn_time   amount  hour  risk_score
T021983    700100603 2025-03-21 00:33:43 30569.23     0    0.803965
T022245    700100603 2025-03-21 23:32:23 27943.44    23    0.796965
T047252    700100333 2025-06-20 00:47:52 32363.89     0    0.796655
T001671    700100873 2025-01-07 04:32:22 76778.30     4    0.796655
T035495    700100616 2025-05-08 04:02:40 45947.43     4    0.796117

Model check against known labels:
              precision    recall  f1-score   support

      Normal       0.99      1.00      0.99     48916
       Fraud       0.62      0.31      0.41      1003

    accuracy                           0.98     49919
   macro avg       0.80      0.65      0.70     49919
weighted avg       0.98      0.98      0.98     49919

Charts and flagged list saved in 'outputs/'
